## Part 1: First connection to the model

In order to run a model, you need to follow the same steps just like any program that requires you authentication (think of Microsoft suit that includes word, power point and excel):
1. **Install** - with some installer.
2. **Open** - with double-click.
3. **Login** - with your name and password.

We will do just that - but in python:
Instead of using the mouse > we will type commands
And unstead of calling our software "program" > we will call it a package (like microsoft) that includes libraries (like words, excel etc)

### Step 1: Install

**In the computer:** we download an installer, press double-click and let it run.

**in Python:** we do not have a mouse - so we type `pip install`. We will install 3 packages:
2. Langchain - because we want to use AI capabilities something else created for us when using OpenAI models
3. langchain-openia - beacuse we want to use OpenAI models.

In [ ]:
!pip install -q langchain langchain-openai

### Step 2: Open the program

**In the computer:** We double click an app to open it.

**in Python:** we do not have a mouse - so we write `import`. Note that Python is "lazy" - it won't "open" everything for us. we will need to specially expain what we need to be opened and from where.

In [24]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool

### Step 3: Login

**In the computer:** we login with Username and Password
**In python:** We use API key

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key="Place here your API key")

# Step 4: Use the program

Let's play around with what we installed:
Run the prompt "Introduce yourself - which model are you?" with model `gpt-4o-mini` which doesn't costs much:

In [29]:
response = llm.invoke("Introduce yourself - which model are you?")
print(response.content)

I am based on OpenAI's GPT-3 model, a sophisticated language model designed to understand and generate human-like text. My purpose is to assist with a variety of tasks, including providing information, answering questions, and engaging in conversation. How can I help you today?


Try it yourslef - here is the same code like before: update only the prompt under the prompt content

In [36]:
response = llm.invoke("Introduce yourself - which model are you?")
response.pretty_print()

================================== Ai Message ==================================

I am ChatGPT, based on the GPT-4 architecture developed by OpenAI. I'm designed to assist with a wide range of questions and tasks, providing information, answering queries, and engaging in conversation. How can I help you today?


Check point - is all good? Let's wait for the others...

## Part 2 - Let's create a tool

In this step we will create an imaginary tool: this tool has a made-up name - `mojifuzzle` that gets one of two instructions "star" of "heart", and a number of emojies to create out of them.
If it gets somthing else - it will return a puzzled emoji.

In order to make it run, we will first run this cell that does nothing but to create the tool:

In [ ]:
def mojifuzzle(emoji_name, count):
    if emoji_name == 'heart':
        return '❤️' * count
    elif emoji_name == 'star':
        return '⭐' * count
    else:
        return '🤔'

Let's try it out. Run the following cells:

In [14]:
mojifuzzle('heart', 10)

'❤️❤️❤️❤️❤️❤️❤️❤️❤️❤️'

In [15]:
mojifuzzle('star', 2)

'⭐⭐'

In [16]:
mojifuzzle('moon', 2)

'🤔'

Check point - is all good? Let's wait for the others...

## Part 3: Let's create an AI tool

please mojifuzzle 3 times with a heartWe want our llm to use our tool. For that end, we will write it this prompt: ``.
Run the following cell that asks our model to do it:


In [ ]:
response = llm.invoke("please mojifuzzle 3 times with a heart")
response.pretty_print()

Sure! Here’s "mojifuzzle" with a heart emoji three times:

💖 mojifuzzle 💖 mojifuzzle 💖 mojifuzzle 💖


As you suspect - the LLM improvised... It did the best it could with the available information he had. 
Run it 2-3 more times and see how the response (and the improvizations) changes.

Now let's use our tool.
The first step would be to make it a tool that the AI can use.
For this we need only a slight change:

![Diagram](images/functionToTool.png)

In [33]:
from langchain.tools import tool

@tool
def mojifuzzle(emoji_name: str, count: int) -> str:
    """Create emojis using 'heart' or 'star'."""
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    else:
        return "🤔"


In [42]:
print(mojifuzzle)
print(mojifuzzle.args)
print(mojifuzzle.description)

name='mojifuzzle' description="Create emojis using 'heart' or 'star'." args_schema=<class 'langchain_core.utils.pydantic.mojifuzzle'> func=<function mojifuzzle at 0x000001E2534A3130>
{'emoji_name': {'title': 'Emoji Name', 'type': 'string'}, 'count': {'title': 'Count', 'type': 'integer'}}
Create emojis using 'heart' or 'star'.


Now, let's connect it to out model. In the technical lingo we say "bind":

In [ ]:
llm_with_tools = llm.bind_tools([mojifuzzle])
response = llm_with_tools.invoke("please mojifuzzle a heart three times")
response.pretty_print()

Check point - is all good? Let's wait for the others...

## Part 4 (Optional): How to make the agent use the tool if needed?

For this we need a more suphisticated package - langgraph.

The reason: we want to create some kind of a state graph:

// TODO a place holder for the graph

As before, let's install it. Wh will also install graphic packages that will draw the graph for us named Graphiz:

In [ ]:
!pip install -q langgraph graphviz

"Open it":

In [ ]:
from langgraph.graph import StateGraph, END


Let's just create the graph:

In [ ]:
def call_model(state):
    response = llm.invoke(state["input"])
    return {"response": response}

def run_tool(state):
    tool_call = state["response"].tool_calls[0]
    args = tool_call["args"]
    result = mojifuzzle(**args)
    return {"output": result}

# 4. Build graph
graph = StateGraph(dict)

graph.add_node("model", call_model)
graph.add_node("tool", run_tool)

# flow: model → tool → end
graph.set_entry_point("model")
graph.add_edge("model", "tool")
graph.add_edge("tool", END)

app = graph.compile()

Let's see how the graph looks like:

from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

Now let's activate it, once with a ageneral prompt, and once with a prompt that needs to use the tool:

In [ ]:
result = app.invoke({"input": "please introduce yourself"})

print(result["output"])

In [ ]:
result = app.invoke({"input": "please mojifuzzle 3 times with a heart"})

print(result["output"])

-----------

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent # The "easy" way

# 1. Define the tool (same as yours)
@tool
def mojifuzzle(emoji_name: str, count: int) -> str:
    """Create a sequence of emojis. Use 'heart' or 'star'."""
    if emoji_name == "heart":
        return "❤️" * count
    elif emoji_name == "star":
        return "⭐" * count
    return "🤔"

tools = [mojifuzzle]
llm = ChatOpenAI(model="gpt-4o-mini", api_key="My Key")

# 2. Create the Agent
# This replaces your manual if/else logic entirely.
app = create_react_agent(llm, tools)

# 3. Run it
# The 'messages' list keeps track of the conversation history.
input_data = {"messages": [("user", "please mojifuzzle 3 times with a heart")]}
config = {"configurable": {"thread_id": "1"}}

for event in app.stream(input_data, config, stream_mode="values"):
    # This prints the full conversation steps clearly
    last_message = event["messages"][-1]
    last_message.pretty_print()

C:\Users\User\AppData\Local\Temp\ipykernel_12616\675066203.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app = create_react_agent(llm, tools)


================================ Human Message =================================

please mojifuzzle 3 times with a heart
================================== Ai Message ==================================
Tool Calls:
  mojifuzzle (call_USL2idW3Xd2TNbdXo7CuxpOU)
 Call ID: call_USL2idW3Xd2TNbdXo7CuxpOU
  Args:
    emoji_name: heart
    count: 3
================================= Tool Message =================================
Name: mojifuzzle

❤️❤️❤️
================================== Ai Message ==================================

Here are three hearts: ❤️❤️❤️
